# TorchRef Basic Usage

This notebook demonstrates the basic usage of TorchRef for crystallographic refinement.

## Setup

In [3]:
import torchref
from torchref import ROOT_TORCHREF

# File paths
pdb_file = f'{ROOT_TORCHREF}/example_notebooks/1DAW.pdb'
mtz_file = f'{ROOT_TORCHREF}/example_notebooks/1DAW.mtz'

/tmp/ipykernel_2388482/2666636.py:1: UserWarning: TorchRef auto-configured 4 threads. Set TORCHREF_NUM_THREADS to override.
  import torchref


## Loading Structures

TorchRef provides two model classes:
- `Model`: Basic model for atomic coordinates
- `ModelFT`: Model with structure factor calculation capability

In [4]:
from torchref.model import Model, ModelFT

# Basic model
model = Model().load_pdb(pdb_file)

# Model with structure factor calculation
model_ft = ModelFT().load_pdb(pdb_file)

Loaded 3051 atoms
Loaded 3051 atoms
Parametrization built for 6 unique atom types
MapSymmetry: Using direct indexing (no interpolation) for <gemmi.SpaceGroup("C 1 2 1")>
MapSymmetryDirect initialized for <gemmi.SpaceGroup("C 1 2 1")>
  Number of symmetry operations: 4
  Map shape: (360, 144, 108)
  ✓ Using direct indexing (no interpolation)


## Loading Reflection Data

In [5]:
from torchref.io import ReflectionData

reflection_data = ReflectionData().load_mtz(mtz_file)

FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged


## Scaling

The `Scaler` class handles bulk solvent correction and scale factor optimization.

In [6]:
from torchref.scaling import Scaler

scaler = Scaler(model=model_ft, data=reflection_data)
scaler.initialize()

print(f"Before refinement: Rwork={scaler.rfactor()[0]:.4f}, Rfree={scaler.rfactor()[1]:.4f}")

scaler.refine_lbfgs()

print(f"After refinement:  Rwork={scaler.rfactor()[0]:.4f}, Rfree={scaler.rfactor()[1]:.4f}")

Initialized Scaler with 20 bins.
Calculating initial scale factors using 20 bins.


/das/work/p17/p17490/CONDA/torchref/lib/python3.11/site-packages/torch/masked/maskedtensor/core.py:247: UserWarning: It is not recommended to create a MaskedTensor with a tensor that requires_grad. To avoid this, you can use data.detach().clone()
  return MaskedTensor(data, mask)


Before refinement: Rwork=0.2882, Rfree=0.3354
Refining scales with LBFGS...
Scale refinement complete. rwork: 0.2094, rfree: 0.2745

Final Scale Parameters: 
  log_scale: tensor([-5.9869, -5.9039, -5.8755, -5.8215, -5.8081, -5.7792, -5.7399, -5.7038,
        -5.7011, -5.6607, -5.6371, -5.5920, -5.5365, -5.4917, -5.4496, -5.3813,
        -5.3029, -5.2917, -5.3083, -5.3378])
  U: tensor([-0.2675, -0.1773, -0.0918, -0.0032, -0.1592, -0.0038])
  solvent.log_k_solvent: -0.9701245427131653
  solvent.b_solvent: 46.05974197387695
  solvent.phase_offset: -0.0014240611344575882


/das/work/p17/p17490/CONDA/torchref/lib/python3.11/site-packages/torch/masked/maskedtensor/core.py:247: UserWarning: It is not recommended to create a MaskedTensor with a tensor that requires_grad. To avoid this, you can use data.detach().clone()
  return MaskedTensor(data, mask)


After refinement:  Rwork=0.2094, Rfree=0.2745


## Setting Up Refinement, low level functions

The `LBFGSRefinement` class provides a complete refinement workflow.
This is demonstrated in the code examples. 

Here we show how the refinement is done at a lower level.

In [7]:
from torchref.refinement import LBFGSRefinement

refinement = LBFGSRefinement(pdb=pdb_file, data_file=mtz_file)

print(f"Initial R-factors: Rwork={refinement.get_rfactor()[0]:.4f}, Rfree={refinement.get_rfactor()[1]:.4f}")

FrenchWilsonModule initialized:
  Reflections: 23356
  Resolution: 2.05 - 69.56 Å
  Space group: <gemmi.SpaceGroup("C 1 2 1")>
  Centric: 1125 (4.8%)
found nan F values:  0
found nan F_sigma values:  0
Suspicious sigma detection: 0/23356 (0.00%) reflections flagged
Loaded 3051 atoms
Parametrization built for 6 unique atom types
MapSymmetry: Using direct indexing (no interpolation) for <gemmi.SpaceGroup("C 1 2 1")>
MapSymmetryDirect initialized for <gemmi.SpaceGroup("C 1 2 1")>
  Number of symmetry operations: 4
  Map shape: (160, 72, 54)
  ✓ Using direct indexing (no interpolation)
Initialized Scaler with 10 bins.
Found 109 link definitions
Built 326 peptide bond restraints
Built 978 peptide angle restraints
Built 326 peptide plane restraints

Building VDW (non-bonded) restraints...
  Built 28724 VDW restraints (all contacts)
Restraints Summary (New Implementation)
CIF file: None
Residue types in dictionary: 21

INTRA-RESIDUE RESTRAINTS:
------------------------------------------------

/das/work/p17/p17490/CONDA/torchref/lib/python3.11/site-packages/torch/masked/maskedtensor/core.py:247: UserWarning: It is not recommended to create a MaskedTensor with a tensor that requires_grad. To avoid this, you can use data.detach().clone()
  return MaskedTensor(data, mask)


## Perturbing the Model

Shake coordinates to simulate a starting model with errors.

In [8]:
refinement.model.shake_coords(0.1)  # Shake by 0.1 Angstroms

print(f"After shaking: Rwork={refinement.get_rfactor()[0]:.4f}, Rfree={refinement.get_rfactor()[1]:.4f}")

After shaking: Rwork=0.2330, Rfree=0.2900


## Loss State and Weights

The `LossState` object manages targets and their weights for refinement.

In [9]:
loss_state = refinement.create_loss_state()
refinement.add_target_info_to_state(loss_state)
refinement.populate_state_meta(loss_state)
refinement.update_weights(loss_state)

print("Weights:")
for name, weight in loss_state.weights.items():
    print(f"  {name}: {weight:.4f}")

print(f"\nTotal loss: {loss_state.aggregate().item():.2f}")

Weights:
  xray: 0.5200
  geometry/bond: 15.9709
  geometry/angle: 3.9377
  geometry/torsion: 1.0873
  geometry/planarity: 37.2188
  geometry/chiral: 375.4997
  geometry/nonbonded: 1.0495
  adp/simu: 2.4221
  adp/locality: 1.2741
  adp/KL: 1.3165
  geometry: 954.5432
  adp: 954.5432

Total loss: 3104928.50


## Running Refinement (CPU)

In [10]:
parameters = refinement.parameters()
refinement._optimize_lbfgs(loss_state, parameters, max_iter=100, nsteps=1)

print(f"After refinement: Rwork={refinement.get_rfactor()[0]:.4f}, Rfree={refinement.get_rfactor()[1]:.4f}")

After refinement: Rwork=0.2227, Rfree=0.2782


In [11]:
print(refinement.collect_metrics())

{'rwork': 0.22273580729961395, 'rfree': 0.27824968099594116, 'rfree_gap': 0.05551387369632721, 'geometry': {'bond': {'loss': -3.470195770263672, 'n': 2837, 'rms_delta': 0.0012474600225687027, 'rms_z': 0.09052692353725433, 'mean_sigma': 0.012758336029946804}, 'angle': {'loss': -2.1545510292053223, 'n': 3828, 'rms_delta': 1.8152810335159302, 'rms_z': 0.8721813559532166, 'mean_sigma': 1.885767936706543}, 'torsion': {'loss': 0.6177268028259277, 'n': 1074, 'rms_delta': 17.244342803955078, 'rms_z': 1.7284802198410034, 'mean_sigma': 9.892923355102539}, 'planarity': {'loss': -2.321131467819214, 'n': 3149, 'rms_delta': 0.010351715609431267, 'rms_z': 0.10473863035440445, 'mean_sigma': 0.05312797799706459}, 'chiral': {'loss': -0.6904860734939575, 'n': 407, 'rms_delta': 0.0010337900603190064, 'rms_z': 0.005168950650840998, 'mean_sigma': 0.20000001788139343}, 'nonbonded': {'loss': 0.04997052997350693, 'n': 28724, 'n_violations': 2482, 'rms_violation': 0.37563401460647583, 'max_violation': 3.4869971

## GPU Acceleration

Move the refinement to GPU for faster computation.

In [12]:
import torch

if torch.cuda.is_available():
    refinement.cuda()
    loss_state.cuda()
    
    # Run refinement on GPU
    parameters = refinement.parameters()
    refinement._optimize_lbfgs(loss_state, parameters, max_iter=100, nsteps=1)
    
    print(f"After GPU refinement: Rwork={refinement.get_rfactor()[0]:.4f}, Rfree={refinement.get_rfactor()[1]:.4f}")
else:
    print("CUDA not available")

Model moved to device: cuda
After GPU refinement: Rwork=0.2293, Rfree=0.2801
